# Part 1
https://huggingface.co/docs/transformers/tasks/token_classification

In [1]:
import torch

device = torch.device("cuda:0")

In [2]:
from datasets import load_dataset

wnut = load_dataset("wnut_17")

In [3]:
import pandas as pd
pd.DataFrame(wnut["train"][0])

,id,tokens,ner_tags
0,0,@paulwalk,0
1,0,It,0
2,0,'s,0
3,0,the,0
4,0,view,0
5,0,from,0
6,0,where,0
7,0,I,0
8,0,'m,0
9,0,living,0


In [4]:
label_list = wnut["train"].features[f"ner_tags"].feature.names

In [5]:
label_list

['O',
 'B-corporation',
 'I-corporation',
 'B-creative-work',
 'I-creative-work',
 'B-group',
 'I-group',
 'B-location',
 'I-location',
 'B-person',
 'I-person',
 'B-product',
 'I-product']

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

In [7]:
example = wnut["train"][0]
tokenized_input = tokenizer(example["tokens"], is_split_into_words=True)
tokens = tokenizer.convert_ids_to_tokens(tokenized_input["input_ids"])

*  is_split_into_words (bool, optional, defaults to False)
    - Whether or not the input is already pre-tokenized (e.g., split into words). If set to True, the tokenizer assumes the input is already split into words (for instance, by splitting it on whitespace) which it will tokenize. This is useful for NER or token classification.
    - https://huggingface.co/transformers/v4.7.0/main_classes/tokenizer.html

In [8]:
tokenized_input

{'input_ids': [101, 1030, 2703, 17122, 2009, 1005, 1055, 1996, 3193, 2013, 2073, 1045, 1005, 1049, 2542, 2005, 2048, 3134, 1012, 3400, 2110, 2311, 1027, 9686, 2497, 1012, 3492, 2919, 4040, 2182, 2197, 3944, 1012, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [9]:
tokens

['[CLS]',
 '@',
 'paul',
 '##walk',
 'it',
 "'",
 's',
 'the',
 'view',
 'from',
 'where',
 'i',
 "'",
 'm',
 'living',
 'for',
 'two',
 'weeks',
 '.',
 'empire',
 'state',
 'building',
 '=',
 'es',
 '##b',
 '.',
 'pretty',
 'bad',
 'storm',
 'here',
 'last',
 'evening',
 '.',
 '[SEP]']

In [10]:
len(example["tokens"])

27

In [11]:
len(tokens)

34

In [12]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples[f"ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  # Map tokens to their respective word.
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:  # Set the special tokens to -100.
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:  # Only label the first token of a given word.
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

* truncation
  - Activates and controls truncation. Accepts the following values:
      - ``True or 'longest_first'``: Truncate to a maximum length specified with the argument max_length or to the maximum acceptable input length for the model if that argument is not provided. This will truncate token by token, removing a token from the longest sequence in the pair if a pair of sequences (or a batch of pairs) is provided.
      - ``'only_first'``: Truncate to a maximum length specified with the argument max_length or to the maximum acceptable input length for the model if that argument is not provided. This will only truncate the first sequence of a pair if a pair of sequences (or a batch of pairs) is provided.
      - ``'only_second'``: Truncate to a maximum length specified with the argument max_length or to the maximum acceptable input length for the model if that argument is not provided. This will only truncate the second sequence of a pair if a pair of sequences (or a batch of pairs) is provided.
      - ``False or 'do_not_truncate' (default)``: No truncation (i.e., can output batch with sequence lengths greater than the model maximum admissible input size).
https://huggingface.co/transformers/v4.7.0/main_classes/tokenizer.html

In [13]:
tokenized_wnut = wnut.map(tokenize_and_align_labels, batched=True)

batched=True to process multiple elements of the dataset at once:

In [14]:
tokenized_wnut

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 3394
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1009
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1287
    })
})

In [15]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

 It’s more efficient to dynamically pad the sentences to the longest length in a batch during collation, instead of padding the whole dataset to the maximum length.

In [16]:
import evaluate, seqeval

seqeval = evaluate.load("seqeval")

In [17]:
import numpy as np

labels = [label_list[i] for i in example[f"ner_tags"]]


def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [18]:
id2label = {
    0: "O",
    1: "B-corporation",
    2: "I-corporation",
    3: "B-creative-work",
    4: "I-creative-work",
    5: "B-group",
    6: "I-group",
    7: "B-location",
    8: "I-location",
    9: "B-person",
    10: "I-person",
    11: "B-product",
    12: "I-product",
}
label2id = {
    "O": 0,
    "B-corporation": 1,
    "I-corporation": 2,
    "B-creative-work": 3,
    "I-creative-work": 4,
    "B-group": 5,
    "I-group": 6,
    "B-location": 7,
    "I-location": 8,
    "B-person": 9,
    "I-person": 10,
    "B-product": 11,
    "I-product": 12,
}

In [19]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert/distilbert-base-uncased", num_labels=13, id2label=id2label, label2id=label2id
)

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [20]:
model = model.to(device)

In [23]:
training_args = TrainingArguments(
    output_dir="my_model",
    report_to="none",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_dir="./logs",
    push_to_hub=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_wnut["train"],
    eval_dataset=tokenized_wnut["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

/home/gusohseb@GU.GU.SE/.local/lib/python3.12/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
/home/gusohseb@GU.GU.SE/.local/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.239500,0.309251,0.398693,0.169601,0.237971,0.934932
2,0.149900,0.305839,0.569794,0.230769,0.328496,0.938609
3,0.109800,0.272927,0.581034,0.312326,0.406269,0.942884
4,0.081400,0.279745,0.556061,0.340130,0.422082,0.944466
5,0.068100,0.278511,0.538881,0.366080,0.435982,0.945364
6,0.053300,0.287253,0.549101,0.367933,0.440622,0.945706
7,0.045400,0.295128,0.543860,0.373494,0.442857,0.946048
8,0.041200,0.306382,0.555714,0.360519,0.437324,0.945449
9,0.037500,0.302315,0.567308,0.382762,0.457111,0.946304
10,0.034500,0.304025,0.556157,0.380908,0.452145,0.946090


/home/gusohseb@GU.GU.SE/.local/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/gusohseb@GU.GU.SE/.local/lib/python3.12/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
/home/gusohseb@GU.GU.SE/.local/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/home/gusohseb@GU.GU.SE/.local/lib/python3.12/site-packages/torch/nn/

TrainOutput(global_step=540, training_loss=0.08604586389329698, metrics={'train_runtime': 403.7911, 'train_samples_per_second': 84.053, 'train_steps_per_second': 1.337, 'total_flos': 518727058938300.0, 'train_loss': 0.08604586389329698, 'epoch': 10.0})

In [44]:
#trainer.push_to_hub()

In [45]:
text = "The Golden State Warriors are an American professional basketball team based in San Francisco."

In [46]:
model_dir = "my_model"

In [47]:
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForTokenClassification.from_pretrained(model_dir)

In [48]:
model = model.to(device)

In [49]:
inputs = tokenizer(text, return_tensors="pt", truncation=True, is_split_into_words=False)
inputs = {k: v.to(device) for k, v in inputs.items()}

In [50]:
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

In [52]:
import torch.nn.functional as F

predictions = torch.argmax(F.softmax(logits, dim=2), dim=2)[0].cpu().numpy()
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
labels = [model.config.id2label[pred] for pred in predictions]

In [53]:
for token, label in zip(tokens, labels):
    print(f"{token}: {label}")

[CLS]: O
the: B-group
golden: B-location
state: I-location
warriors: B-group
are: O
an: O
american: O
professional: O
basketball: O
team: O
based: O
in: O
san: B-location
francisco: B-location
.: O
[SEP]: O
